# 🧪 Experimentation Framework
### Portfolio Project 5 of 5 · Jordan Shamukiga

> **A production-grade, reusable A/B testing toolkit** for product and analytics teams.  
> Covers: Power Analysis · Sequential Testing · Multiple Corrections · Bayesian A/B · Project 2 Retrospective

---

**Portfolio:** [datascienceportfol.io/jordanshamu](https://datascienceportfol.io/jordanshamu)  
**GitHub:** [github.com/jordanshamu](https://github.com/jordanshamu)

---

## Portfolio Context

| # | Project | Status |
|---|---------|--------|
| 1 | Healthcare Readmission Prediction | ✅ Complete |
| 2 | [Marketing A/B Test Analysis](https://github.com/jordanshamu/Marketing-A-B-Test-Analysis) | ✅ Complete |
| 3 | Customer Segmentation & Cohort Analysis | ✅ Complete |
| 4 | Customer Churn Prediction | ✅ Complete |
| **5** | **Experimentation Framework** | ✅ **Complete** |

---


## 0. Environment Setup

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from stats_utils import (
    sample_size_proportion, sample_size_ttest, power_curve, sample_size_curve,
    sprt_boundaries, sprt_llr, obrien_fleming_boundaries, simulate_sequential_test,
    multiple_testing_summary, bonferroni, holm_bonferroni, benjamini_hochberg,
    bayesian_ab_summary, frequentist_ab_summary, beta_posterior,
    prob_b_beats_a, expected_loss,
)

# ── Plot styling ────────────────────────────────────────────────────────────
PALETTE = ["#2C3E50","#E74C3C","#27AE60","#F39C12","#8E44AD","#16A085"]
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "#F8F9FA",
    "axes.grid": True, "grid.alpha": 0.4,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "font.size": 11,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.labelsize": 11,
})
VIZ_DIR = os.path.join(os.path.dirname(os.getcwd()), "visualizations")
os.makedirs(VIZ_DIR, exist_ok=True)

def save(name):
    path = os.path.join(VIZ_DIR, name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Saved → {name}")

print("✅ Setup complete")


---
## 1. Power & Sample Size Calculator

> **Business framing:** Running an underpowered experiment is like flipping a coin twice
> and concluding it's biased. We waste traffic, delay decisions, and—most dangerously—
> publish null results that are actually false negatives.
>
> This calculator answers: *"How many users do we need before we start?"*

### 1.1 The Core Concepts

| Term | Statistical Definition | Business Translation |
|------|----------------------|---------------------|
| **α (significance level)** | P(reject H0 \| H0 true) | False positive rate — how often we ship something that doesn't work |
| **1-β (power)** | P(reject H0 \| H1 true) | True positive rate — how often we detect an effect that's real |
| **MDE** | Minimum Detectable Effect | The smallest lift worth caring about from a revenue perspective |
| **n per variant** | Required sample size | Users needed *per arm* before the test is conclusive |


In [ ]:
# ── 1.2 Proportion Test (conversion rates) ───────────────────────────────────
# Scenario: our checkout converts at 10%; we care about lifts ≥ 2 pp
result = sample_size_proportion(p_baseline=0.10, mde=0.02, alpha=0.05, power=0.80)

print("📊 Sample Size: Two-Proportion Z-Test")
print("=" * 45)
for k, v in result.items():
    print(f"  {k:<22} {v}")


In [ ]:
# ── 1.3 Continuous Metric T-Test ─────────────────────────────────────────────
# Scenario: average session duration = 3 min, std = 1.5 min; MDE = 15 sec
result_t = sample_size_ttest(mean_baseline=180, std_baseline=90, mde=15,
                              alpha=0.05, power=0.80)

print("📊 Sample Size: Independent-Samples T-Test")
print("=" * 45)
for k, v in result_t.items():
    print(f"  {k:<22} {v}")


### 1.4 Power Curve — How Power Changes With Sample Size & MDE

> **Key insight:** Power is not binary. The chart below shows that increasing n from
> 1,000 to 5,000 per variant dramatically expands the range of effects we can detect.
> Plotting this *before* launching prevents underpowered tests.


In [ ]:
mde_range = np.linspace(0.005, 0.060, 100)
n_scenarios = [500, 1000, 2000, 5000, 10000]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Power curves at different n
ax = axes[0]
for i, n in enumerate(n_scenarios):
    pwr = power_curve(0.10, mde_range, n)
    ax.plot(mde_range*100, pwr, color=PALETTE[i], lw=2, label=f"n={n:,}")
ax.axhline(0.80, color="grey", ls="--", lw=1.2, alpha=0.7, label="80% power target")
ax.axhline(0.50, color="grey", ls=":", lw=1.0, alpha=0.5)
ax.set_xlabel("Minimum Detectable Effect (percentage points)")
ax.set_ylabel("Statistical Power (1-β)")
ax.set_title("Power Curves by Sample Size")
ax.legend(fontsize=9, framealpha=0.9)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y:.0%}"))

# Right: Required n at different MDEs (fixed α=0.05, power=0.80)
ax2 = axes[1]
n_required = sample_size_curve(0.10, mde_range)
ax2.fill_between(mde_range*100, n_required, alpha=0.15, color=PALETTE[0])
ax2.plot(mde_range*100, n_required, color=PALETTE[0], lw=2.5)
ax2.axvline(2.0, color=PALETTE[1], ls="--", lw=1.5, label="2pp MDE → 3,842/variant")
ax2.axhline(3842, color=PALETTE[1], ls="--", lw=1.5)
ax2.set_xlabel("Minimum Detectable Effect (percentage points)")
ax2.set_ylabel("Required n per Variant")
ax2.set_title("Required Sample Size vs. MDE\n(baseline=10%, α=0.05, power=80%)")
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{int(y):,}"))

plt.suptitle("Power Analysis Dashboard", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
save("01_power_curves.png")
plt.show()
print("\n💡 Takeaway: Detecting a 1pp lift needs ~14,000 users/variant; a 4pp lift only ~2,400.")


In [ ]:
# ── 1.5 Scenario Comparison Table ────────────────────────────────────────────
scenarios = [
    ("High-traffic, small MDE",  0.10, 0.01, 0.05, 0.80),
    ("Standard product test",    0.10, 0.02, 0.05, 0.80),
    ("Quick-win test",           0.10, 0.04, 0.05, 0.80),
    ("High-stakes, 90% power",   0.10, 0.02, 0.05, 0.90),
    ("Strict α=0.01",            0.10, 0.02, 0.01, 0.80),
    ("Low-baseline metric",      0.03, 0.01, 0.05, 0.80),
]

rows = []
for name, pb, mde, a, pw in scenarios:
    r = sample_size_proportion(pb, mde, a, pw)
    rows.append({"Scenario": name, "Baseline": f"{pb:.0%}", "MDE": f"+{mde:.0%}",
                 "α": a, "Power": pw, "n/variant": f"{r['n_per_variant']:,}",
                 "Total n": f"{r['total_n']:,}"})

df_scen = pd.DataFrame(rows)
print("📋 Sample Size Scenario Comparison")
print(df_scen.to_string(index=False))


---
## 2. Sequential Analysis / Early Stopping

> **Business framing:** Experiment dashboards create a daily temptation: if results look
> good at Day 5, why wait until Day 14? The problem is that every peek is a chance to
> make a false positive error. Run 10 peeks at α=0.05 and your actual false positive
> rate jumps well above 5%.
>
> Sequential analysis provides **valid stopping rules** that preserve your Type I error
> rate while allowing early termination.

### 2.1 The Two Methods

| Method | Best For | Mechanism |
|--------|---------|-----------|
| **SPRT** (Sequential Probability Ratio Test) | Continuous monitoring; testing a point alternative H1 | Likelihood ratio crosses Wald boundaries |
| **O'Brien-Fleming** | Pre-planned interim looks; clinical-trial style governance | Alpha is spent conservatively at early looks, normally at final |


In [ ]:
# ── 2.2 O'Brien-Fleming Boundaries ───────────────────────────────────────────
n_looks = 10
of_bounds = obrien_fleming_boundaries(n_looks, alpha=0.05)
z_naive   = stats.norm.ppf(0.975)   # standard two-tailed z for α=0.05

print("O'Brien-Fleming Boundaries (10 interim looks, α=0.05)")
print(f"{'Look':>5}  {'Fraction':>9}  {'OBF z-boundary':>15}  {'Naive z-boundary':>17}  {'OBF α_spent':>12}")
print("-"*65)
for i, (b, frac) in enumerate(zip(of_bounds, np.linspace(0.1,1.0,n_looks))):
    alpha_spent = 2 * (1 - stats.norm.cdf(b))
    print(f"{i+1:>5}  {frac:>9.1%}  {b:>15.3f}  {z_naive:>17.3f}  {alpha_spent:>12.4f}")


In [ ]:
# ── 2.3 Simulate: Null Scenario (no real effect) — show peeking inflation ───
sim_null = simulate_sequential_test(
    p_control=0.10, p_treatment=0.10,   # TRUE NULL: no effect
    n_total=10_000, alpha=0.05, n_looks=10, seed=7
)

sim_effect = simulate_sequential_test(
    p_control=0.10, p_treatment=0.13,   # TRUE EFFECT: +3pp
    n_total=10_000, alpha=0.05, n_looks=10, seed=42
)

for label, sim in [("🚫 No True Effect (H0 true)", sim_null),
                   ("✅ Real Effect +3pp (H1 true)", sim_effect)]:
    print(f"\n{label}")
    print(f"  Naive peeker stopped early : {sim['naive_stopped']}")
    print(f"  OBF stopped early          : {sim['obf_stopped']}")


In [ ]:
# ── 2.4 Visualise sequential monitoring ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sim, title in zip(
    axes,
    [sim_null, sim_effect],
    ["Scenario A: No True Effect (H₀ True)", "Scenario B: Real +3 pp Effect (H₁ True)"]
):
    df = sim["results"]
    x  = df["look"].values

    ax.fill_between(x, df["obf_boundary"], 10, alpha=0.08, color=PALETTE[2], label="OBF rejection zone")
    ax.fill_between(x, -df["obf_boundary"], -10, alpha=0.08, color=PALETTE[2])

    ax.plot(x, df["naive_boundary"], color=PALETTE[1], ls="--", lw=1.5, label="Naive boundary (±1.96)")
    ax.plot(x, -df["naive_boundary"], color=PALETTE[1], ls="--", lw=1.5)
    ax.plot(x, df["obf_boundary"], color=PALETTE[2], lw=2, label="O'Brien-Fleming boundary")
    ax.plot(x, -df["obf_boundary"], color=PALETTE[2], lw=2)
    ax.plot(x, df["z_stat"], color=PALETTE[0], lw=2.5, marker="o", ms=5, label="Observed z-statistic")

    # Mark stopping points
    ns = sim["naive_stopped"]
    os_ = sim["obf_stopped"]
    if ns:
        ax.axvline(ns["look"], color=PALETTE[1], ls=":", alpha=0.8)
        ax.annotate(f"Naive stops\nLook {ns['look']}",
                    xy=(ns["look"], ns["z"]), xytext=(ns["look"]+0.5, ns["z"]+0.5),
                    fontsize=8, color=PALETTE[1],
                    arrowprops=dict(arrowstyle="->", color=PALETTE[1]))
    if os_:
        ax.axvline(os_["look"], color=PALETTE[2], ls=":", alpha=0.8)
        ax.annotate(f"OBF stops\nLook {os_['look']}",
                    xy=(os_["look"], os_["z"]), xytext=(os_["look"]-2.5, os_["z"]-1.0),
                    fontsize=8, color=PALETTE[2],
                    arrowprops=dict(arrowstyle="->", color=PALETTE[2]))

    ax.set_xlim(0.5, n_looks+0.5)
    ax.set_ylim(-5, 5)
    ax.set_xticks(x)
    ax.set_xlabel("Interim Look Number")
    ax.set_ylabel("z-statistic")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper right")

plt.suptitle("Sequential Monitoring: Naive Peeking vs. O'Brien-Fleming", fontsize=14, fontweight="bold")
plt.tight_layout()
save("02_sequential_analysis.png")
plt.show()


In [ ]:
# ── 2.5 SPRT demo — continuously stream data and watch likelihood ratio ───────
rng = np.random.default_rng(99)
n_stream = 3000
p0, p1 = 0.10, 0.13    # H0: rate=10%  H1: rate=13%  (true: 13%)
lower, upper = sprt_boundaries(alpha=0.05, beta=0.20)

x_t = rng.binomial(1, 0.13, n_stream)   # true treatment = 13%
llr_trace = []
cumsum = 0
for i, obs in enumerate(x_t):
    cumsum += (obs * np.log(p1/p0) + (1-obs) * np.log((1-p1)/(1-p0)))
    llr_trace.append(cumsum)
    if cumsum >= upper or cumsum <= lower:
        stopped_at = i + 1
        break
else:
    stopped_at = n_stream

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(1, len(llr_trace)+1), llr_trace, color=PALETTE[0], lw=1.8, label="Cumulative LLR (Λ)")
ax.axhline(upper, color=PALETTE[2], ls="--", lw=1.8, label=f"Reject H₀ boundary ({upper:.2f})")
ax.axhline(lower, color=PALETTE[1], ls="--", lw=1.8, label=f"Accept H₀ boundary ({lower:.2f})")
ax.axhline(0, color="grey", ls="-", lw=0.8, alpha=0.4)
if stopped_at < n_stream:
    ax.axvline(stopped_at, color=PALETTE[2], ls=":", lw=2)
    ax.annotate(f"SPRT stops at n={stopped_at}\n(Reject H₀)",
                xy=(stopped_at, upper*0.9),
                xytext=(stopped_at+80, upper*0.6),
                fontsize=9, color=PALETTE[2],
                arrowprops=dict(arrowstyle="->", color=PALETTE[2]))
ax.set_xlabel("Users Observed (streaming)")
ax.set_ylabel("Log-Likelihood Ratio (Λ)")
ax.set_title("SPRT: Sequential Probability Ratio Test
(True p_treatment=13%, H1: p=13% vs H0: p=10%)")
ax.legend(fontsize=9)
plt.tight_layout()
save("03_sprt.png")
plt.show()
print(f"\nSPRT stopped at n = {stopped_at:,} (vs full n = {n_stream:,})")
print(f"Efficiency gain: {(1 - stopped_at/n_stream):.0%} fewer observations needed")


---
## 3. Multiple Testing Corrections

> **Business framing:** If you run 20 simultaneous tests at α=0.05, expected false
> discoveries = 20 × 0.05 = **1 false positive guaranteed**. Over a year of product
> experimentation, this compounds into a meaningless shipped-feature backlog.
>
> The three corrections below offer different trade-offs between **controlling false
> positives** and **retaining power to detect true effects**.

### 3.1 The Three Methods

| Method | Controls | Trade-off | When to Use |
|--------|---------|-----------|-------------|
| **Bonferroni** | FWER (any false positive) | Most conservative; low power | Safety features, legal/compliance |
| **Holm-Bonferroni** | FWER | Same guarantee, uniformly more powerful | Standard experimentation default |
| **Benjamini-Hochberg** | FDR (expected proportion of false discoveries) | Most permissive; accepts some false discoveries | Exploratory feature discovery |


In [ ]:
# ── 3.2 Simulate 20 simultaneous hypotheses ───────────────────────────────────
rng = np.random.default_rng(123)
n_tests     = 20
n_true_null = 14   # 14 tests where H0 is true (no effect)
n_true_alt  = 6    # 6 tests where H1 is true (real effect, small)

# Generate p-values: 14 under H0 (uniform), 6 under H1 (small effects)
p_null = rng.uniform(0, 1, n_true_null)
p_alt  = rng.beta(1, 10, n_true_alt)        # skewed toward 0 (real effects)
p_vals = np.concatenate([p_null, p_alt])
truth  = ["H0 (null)"] * n_true_null + ["H1 (effect)"] * n_true_alt
labels = [f"Test {i+1:02d}" for i in range(n_tests)]

df_mt = multiple_testing_summary(p_vals, alpha=0.05, labels=labels)
df_mt["True State"] = truth

print("Multiple Testing Summary (20 simultaneous tests, α = 0.05)")
print(df_mt[["Hypothesis","Raw p","True State",
             "Reject (raw)","Reject – Bonferroni",
             "Reject – Holm","Reject – BH FDR"]].to_string(index=False))


In [ ]:
# ── 3.3 Visualise the comparison ─────────────────────────────────────────────
order = np.argsort(p_vals)
sorted_p  = p_vals[order]
sorted_lbl = np.array(labels)[order]
sorted_truth = np.array(truth)[order]

bon  = bonferroni(p_vals)
holm = holm_bonferroni(p_vals)
bh   = benjamini_hochberg(p_vals)

fig, ax = plt.subplots(figsize=(14, 6))
y = np.arange(n_tests)[::-1]

colors_truth = [PALETTE[2] if t=="H1 (effect)" else PALETTE[0] for t in sorted_truth]
ax.barh(y, sorted_p, color=[PALETTE[2] if t=="H1 (effect)" else "#BDC3C7"
                              for t in sorted_truth], alpha=0.6, height=0.5)

# Mark rejections with symbols
methods_rej = [
    ("Raw (uncorrected)", 0.05,             "x", PALETTE[1], 14),
    ("Bonferroni",        bon["adjusted_p"][order],  "D", PALETTE[0], 12),
    ("Holm-Bonferroni",   holm["adjusted_p"][order], "s", PALETTE[3], 10),
    ("BH-FDR",            bh["adjusted_p"][order],   "o", PALETTE[2], 8),
]
for mlabel, adj, marker, col, ms in methods_rej:
    thresh = 0.05 if isinstance(adj, float) else None
    if thresh:
        rej_mask = sorted_p <= thresh
    else:
        rej_mask = adj <= 0.05
    ax.scatter([-0.01]*rej_mask.sum(), y[rej_mask], marker=marker,
               color=col, s=ms**2, zorder=5, label=f"✓ {mlabel}")

ax.axvline(0.05, color=PALETTE[1], ls="--", lw=1.5, alpha=0.8, label="α = 0.05")
ax.set_yticks(y)
ax.set_yticklabels([f"{l} ({t})" for l,t in zip(sorted_lbl, sorted_truth)], fontsize=8)
ax.set_xlabel("p-value")
ax.set_title("Multiple Testing Corrections: What Each Method Rejects\n"
             "(green = true effect, grey = true null)")
ax.legend(fontsize=8, loc="lower right")
ax.set_xlim(-0.05, 1.0)
plt.tight_layout()
save("04_multiple_testing.png")
plt.show()


In [ ]:
# ── 3.4 Summary table ─────────────────────────────────────────────────────────
# How many true H1s did each method catch? How many false positives?
is_h1 = np.array(truth) == "H1 (effect)"

summary_rows = []
for name, rej in [
    ("Uncorrected (α=0.05)",  p_vals <= 0.05),
    ("Bonferroni",            bon["rejected"]),
    ("Holm-Bonferroni",       holm["rejected"]),
    ("BH-FDR",                bh["rejected"]),
]:
    tp = int(( rej &  is_h1).sum())
    fp = int(( rej & ~is_h1).sum())
    fn = int((~rej &  is_h1).sum())
    summary_rows.append({"Method": name, "True Positives": tp,
                          "False Positives (shipped duds)": fp,
                          "Missed Effects": fn,
                          "Total Rejected": int(rej.sum())})

print(pd.DataFrame(summary_rows).to_string(index=False))
print()
print("💡 BH-FDR catches the most true effects while keeping false positives controlled.")
print("   Bonferroni is safest but misses more real effects.")


---
## 4. Bayesian A/B Testing

> **Business framing:** Frequentist statistics answers: *"Is this result unlikely by
> chance?"* Bayesian statistics answers: *"What do we now believe about conversion
> rates, and what's the expected cost of making the wrong call?"*
>
> For product decisions, the second question is more useful.

### 4.1 The Beta-Binomial Model

We place a **Beta prior** on the true conversion rate (uninformative: Beta(1,1)),
observe conversion data, and update to a **Beta posterior** via conjugate update:

```
Prior:      p ~ Beta(α₀, β₀)
Likelihood: X ~ Binomial(n, p)
Posterior:  p | X ~ Beta(α₀ + X, β₀ + n - X)
```

This gives us a full distribution over the true rate — not just a point estimate.


In [ ]:
# ── 4.2 Simulate an experiment ────────────────────────────────────────────────
np.random.seed(42)
N_C, N_T   = 5_000, 5_000
P_C, P_T   = 0.10, 0.115   # +1.5pp true effect

X_C = np.random.binomial(N_C, P_C)
X_T = np.random.binomial(N_T, P_T)

print(f"Simulated Experiment Results")
print(f"  Control  : {X_C:,}/{N_C:,} conversions ({X_C/N_C:.3%})")
print(f"  Treatment: {X_T:,}/{N_T:,} conversions ({X_T/N_T:.3%})")
print(f"  True lift: +{P_T - P_C:.2%}")


In [ ]:
# ── 4.3 Frequentist result ────────────────────────────────────────────────────
freq = frequentist_ab_summary(N_C, X_C, N_T, X_T)
print("\n📊 Frequentist (z-test) Results")
print("=" * 40)
for k, v in freq.items():
    print(f"  {k:<30} {v}")


In [ ]:
# ── 4.4 Bayesian result ───────────────────────────────────────────────────────
bayes = bayesian_ab_summary(N_C, X_C, N_T, X_T)
print("\n🎲 Bayesian Results")
print("=" * 40)
skip = {"posterior_control","posterior_treatment"}
for k, v in bayes.items():
    if k not in skip:
        print(f"  {k:<38} {v}")


In [ ]:
# ── 4.5 Visualise posterior distributions ────────────────────────────────────
a_c, b_c = bayes["posterior_control"]
a_t, b_t = bayes["posterior_treatment"]

x = np.linspace(0.04, 0.16, 1000)
pdf_c = stats.beta.pdf(x, a_c, b_c)
pdf_t = stats.beta.pdf(x, a_t, b_t)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Posterior distributions
ax = axes[0]
ax.fill_between(x, pdf_c, alpha=0.3, color=PALETTE[0], label="Control posterior")
ax.fill_between(x, pdf_t, alpha=0.3, color=PALETTE[2], label="Treatment posterior")
ax.plot(x, pdf_c, color=PALETTE[0], lw=2)
ax.plot(x, pdf_t, color=PALETTE[2], lw=2)

# Credible intervals
ci_c = stats.beta.ppf([0.025, 0.975], a_c, b_c)
ci_t = stats.beta.ppf([0.025, 0.975], a_t, b_t)
ax.axvline(stats.beta.mean(a_c, b_c), color=PALETTE[0], ls="--", lw=1.5, alpha=0.8)
ax.axvline(stats.beta.mean(a_t, b_t), color=PALETTE[2], ls="--", lw=1.5, alpha=0.8)

ax.set_xlabel("Conversion Rate (p)")
ax.set_ylabel("Posterior Density")
ax.set_title(f"Posterior Distributions\nP(treatment > control) = {bayes['prob_treatment_wins']:.1%}")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y:.1%}"))

# Right: Posterior of the lift (difference)
rng2 = np.random.default_rng(42)
samples_c = rng2.beta(a_c, b_c, 200_000)
samples_t = rng2.beta(a_t, b_t, 200_000)
diff = samples_t - samples_c

ax2 = axes[1]
ax2.hist(diff, bins=80, color=PALETTE[3], alpha=0.7, density=True)
ax2.axvline(0, color=PALETTE[1], lw=2, ls="--", label="No effect (Δ=0)")
ax2.axvline(diff.mean(), color=PALETTE[0], lw=2, ls="--",
            label=f"Posterior mean Δ = {diff.mean():.3%}")
ci_diff = np.percentile(diff, [2.5, 97.5])
ax2.axvspan(ci_diff[0], ci_diff[1], alpha=0.15, color=PALETTE[3],
            label=f"95% CI: [{ci_diff[0]:.3%}, {ci_diff[1]:.3%}]")

pct_pos = (diff > 0).mean()
ax2.set_xlabel("Lift (p_treatment − p_control)")
ax2.set_ylabel("Density")
ax2.set_title(f"Posterior Distribution of Lift\n{pct_pos:.1%} of probability mass above zero")
ax2.legend(fontsize=8)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y:.2%}"))

plt.suptitle("Bayesian A/B Test: Full Posterior Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
save("05_bayesian_posteriors.png")
plt.show()


In [ ]:
# ── 4.6 Expected Loss — the decision-theoretic criterion ─────────────────────
print("\n💰 Expected Loss Analysis")
print("=" * 50)
print(f"  Expected loss of choosing CONTROL  : {bayes['expected_loss_choose_control']:.5%}")
print(f"  Expected loss of choosing TREATMENT: {bayes['expected_loss_choose_treatment']:.5%}")
print()
print(f"  → Lower expected loss = safer choice")
print(f"  → Recommendation: {bayes['recommendation']}")

print("\n📊 Frequentist vs Bayesian — Side by Side")
print(f"  {'Criterion':<35} {'Frequentist':>15} {'Bayesian':>15}")
print("-"*68)
print(f"  {'Significant?':<35} {'Yes' if freq['significant'] else 'No':>15} {'—':>15}")
print(f"  {'p-value':<35} {str(freq['p_value']):>15} {'—':>15}")
print(f"  {'P(treatment > control)':<35} {'—':>15} {str(bayes['prob_treatment_wins']):>15}")
print(f"  {'Expected loss (choose treatment)':<35} {'—':>15} {bayes['expected_loss_choose_treatment']:>15.5%}")
print(f"  {'Recommendation':<35} {'Ship' if freq['significant'] else 'Hold':>15} {bayes['recommendation']:>15}")


---
## 5. Test Design Templates

Three production-ready documents that make experiment rigour the default, not the
exception. Stored in `templates/` — clone and fill in for each new experiment.

| Template | Purpose | Who Uses It |
|----------|---------|-------------|
| `experiment_brief.md` | Define hypothesis, metrics, design before any code is written | PM + Analyst |
| `preregistration_checklist.md` | Lock in analysis plan; prevent HARKing and p-hacking | Analyst |
| `results_writeup.md` | Standardised format for sharing conclusions with stakeholders | Analyst + PM |

> **Why pre-registration matters:**  
> Without locking in your analysis plan before seeing data, it's easy to
> unconsciously (or consciously) search for a "winning" cut of the data. Pre-registration
> makes the intended analysis explicit and verifiable — the same discipline used in
> clinical trials.


In [ ]:
import os

template_dir = os.path.join(os.path.dirname(os.getcwd()), "templates")
for fname in sorted(os.listdir(template_dir)):
    path = os.path.join(template_dir, fname)
    size = os.path.getsize(path)
    print(f"  📄 {fname:<40}  ({size:,} bytes)")

print("\nAll templates available in templates/ directory.")
print("Clone, fill in, and commit alongside each experiment.")


---
## 6. Project 2 Retrospective — Marketing A/B Test Revisited

> **Context:** [Project 2](https://github.com/jordanshamu/Marketing-A-B-Test-Analysis)
> analysed a marketing A/B test across **588,101 users** and found a statistically
> significant **42.5% conversion lift** (p < 0.001).
>
> This section applies the full experimentation framework retroactively to answer:
> 1. Was the test adequately powered for what was found?
> 2. Could it have been stopped early using valid sequential methods?
> 3. What does the Bayesian posterior show?

### Key Facts from Project 2

| Parameter | Value |
|-----------|-------|
| Total users | 588,101 |
| Control n | ~294,050 |
| Treatment n | ~294,051 |
| Control conversion rate | ~2.93% |
| Treatment conversion rate | ~4.18% |
| Observed lift | +1.25 pp (42.5% relative) |
| Reported p-value | < 0.001 |


In [ ]:
# ── 6.1 Was it adequately powered? ───────────────────────────────────────────
# Reconstruct approximate numbers from reported stats
N_PER_VARIANT = 294_050
P_CTRL   = 0.0293
P_TREAT  = 0.0418
LIFT_ABS = P_TREAT - P_CTRL      # ~1.25 pp

# Required n for this MDE at 80% power
req = sample_size_proportion(p_baseline=P_CTRL, mde=LIFT_ABS, alpha=0.05, power=0.80)
req_90 = sample_size_proportion(p_baseline=P_CTRL, mde=LIFT_ABS, alpha=0.05, power=0.90)

print("📊 Project 2: Sample Size Retrospective")
print(f"  Actual n per variant       : {N_PER_VARIANT:>10,}")
print(f"  Required (α=0.05, 80% pwr) : {req['n_per_variant']:>10,}")
print(f"  Required (α=0.05, 90% pwr) : {req_90['n_per_variant']:>10,}")
print(f"  Overpowered by factor      : {N_PER_VARIANT/req['n_per_variant']:>10.0f}x")
print()
print(f"  → The test ran with {N_PER_VARIANT/req['n_per_variant']:.0f}x the required sample. ")
print(f"    Even a 0.1pp MDE would have been detectable.")

# What MDE could we detect at 80% power with n=294,050?
mde_range = np.linspace(0.0001, 0.005, 1000)
pwr = power_curve(P_CTRL, mde_range, N_PER_VARIANT)
detectable_mde = mde_range[np.searchsorted(pwr, 0.80)]
print(f"\n  Smallest detectable MDE at 80% power: {detectable_mde:.4%} ({detectable_mde*100:.2f} pp)")


In [ ]:
# ── 6.2 Sequential stopping: would OBF have allowed early stopping? ───────────
# Simulate the experiment as a stream with 10 interim looks
sim_p2 = simulate_sequential_test(
    p_control=P_CTRL, p_treatment=P_TREAT,
    n_total=N_PER_VARIANT * 2,
    alpha=0.05, n_looks=10, seed=42
)

df_p2 = sim_p2["results"]
print("📊 Project 2: Sequential Analysis (O'Brien-Fleming, 10 looks)")
print(df_p2[["look","n_seen","r_control","r_treatment","z_stat",
              "obf_boundary","obf_reject"]].to_string(index=False))
print()
if sim_p2["obf_stopped"]:
    s = sim_p2["obf_stopped"]
    pct_saved = (1 - s["n"] / (N_PER_VARIANT*2)) * 100
    print(f"  ✅ OBF would have stopped at look {s['look']} (n={s['n']:,})")
    print(f"     Saving {pct_saved:.0f}% of experiment runtime")
else:
    print("  Test ran to completion under OBF boundaries")


In [ ]:
# ── 6.3 Bayesian retrospective ────────────────────────────────────────────────
X_C_P2 = int(N_PER_VARIANT * P_CTRL)
X_T_P2 = int(N_PER_VARIANT * P_TREAT)

bayes_p2 = bayesian_ab_summary(N_PER_VARIANT, X_C_P2, N_PER_VARIANT, X_T_P2)
freq_p2  = frequentist_ab_summary(N_PER_VARIANT, X_C_P2, N_PER_VARIANT, X_T_P2)

print("📊 Project 2: Bayesian Retrospective")
print(f"  P(treatment > control)            : {bayes_p2['prob_treatment_wins']:.6f}")
print(f"  Expected loss (choose treatment)  : {bayes_p2['expected_loss_choose_treatment']:.8f}")
print(f"  Bayesian recommendation           : {bayes_p2['recommendation']}")
print()
print(f"  Frequentist p-value               : {freq_p2['p_value']:.2e}")
print(f"  Frequentist significant           : {freq_p2['significant']}")
print()
print("  → Both methods give the same decision: ship the treatment.")
print("  → Bayesian P(treatment wins) ≈ 1.000000: overwhelming posterior evidence.")
print("  → Expected loss of shipping treatment ≈ 0: no meaningful downside risk.")


In [ ]:
# ── 6.4 Visualise Project 2 posteriors ───────────────────────────────────────
a_c2, b_c2 = bayes_p2["posterior_control"]
a_t2, b_t2 = bayes_p2["posterior_treatment"]

x2 = np.linspace(0.025, 0.050, 1000)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(x2, stats.beta.pdf(x2, a_c2, b_c2), alpha=0.4, color=PALETTE[0])
ax.fill_between(x2, stats.beta.pdf(x2, a_t2, b_t2), alpha=0.4, color=PALETTE[2])
ax.plot(x2, stats.beta.pdf(x2, a_c2, b_c2), color=PALETTE[0], lw=2,
        label=f"Control posterior (mean={P_CTRL:.3%})")
ax.plot(x2, stats.beta.pdf(x2, a_t2, b_t2), color=PALETTE[2], lw=2,
        label=f"Treatment posterior (mean={P_TREAT:.3%})")
ax.set_title("Project 2: Posterior Distributions
(n=294K per arm)")
ax.set_xlabel("Conversion Rate")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y:.2%}"))

# Sequential stopping visualisation
ax2 = axes[1]
ax2.plot(df_p2["look"], df_p2["z_stat"], color=PALETTE[0], lw=2.5, marker="o",
         ms=5, label="z-statistic")
ax2.plot(df_p2["look"], df_p2["obf_boundary"], color=PALETTE[2], lw=2, ls="--",
         label="O'Brien-Fleming boundary")
ax2.plot(df_p2["look"], df_p2["naive_boundary"], color=PALETTE[1], lw=1.5, ls="--",
         label="Naive boundary (±1.96)")
ax2.fill_between(df_p2["look"], df_p2["obf_boundary"],
                  df_p2["z_stat"].clip(lower=df_p2["obf_boundary"]),
                  alpha=0.15, color=PALETTE[2])
if sim_p2["obf_stopped"]:
    ax2.axvline(sim_p2["obf_stopped"]["look"], color=PALETTE[3], ls=":", lw=2,
                label=f"Safe stopping point (look {sim_p2['obf_stopped']['look']})")
ax2.set_title("Project 2: Sequential Monitoring Retrospective")
ax2.set_xlabel("Interim Look")
ax2.set_ylabel("z-statistic")
ax2.legend(fontsize=8)

plt.suptitle("Project 2 Marketing A/B Test — Full Framework Retrospective",
             fontsize=14, fontweight="bold")
plt.tight_layout()
save("06_project2_retrospective.png")
plt.show()

print(f"\n✅ Retrospective Summary")
print(f"   Sample adequacy   : {N_PER_VARIANT/req['n_per_variant']:.0f}x overpowered — test was conclusive by design")
print(f"   Sequential testing: OBF allows early stopping at {sim_p2['obf_stopped']['look'] if sim_p2['obf_stopped'] else 'N/A'}/10 looks")
print(f"   Bayesian verdict  : P(treatment wins) ≈ 1.0 — essentially certain")
print(f"   Both methods agree: Ship the treatment variant.")


---
## 7. Framework Summary & Decision Guide

### When to Use Each Component

| Situation | Component to Use |
|-----------|-----------------|
| Planning a new experiment | Power & Sample Size Calculator |
| Single test, standard runtime | Frequentist z-test / t-test |
| Need to monitor or stop early | Sequential Analysis (O'Brien-Fleming) |
| Continuous monitoring without fixed looks | SPRT |
| Testing multiple metrics or segments | Multiple Testing Correction |
| High-stakes decision requiring probability + risk quantification | Bayesian A/B |
| Documenting an experiment before launch | Pre-registration Checklist |
| Communicating results to stakeholders | Results Write-Up Template |

---

### The Four Most Common Experiment Mistakes — and How This Framework Fixes Them

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| **Underpowered test** | False negatives; wasted traffic | Power calculator before launch |
| **Naive peeking** | Inflated false positive rate | Sequential boundaries (OBF / SPRT) |
| **Ignoring multiple comparisons** | Shipped features that don't work | Holm or BH-FDR correction |
| **Binary p-value thinking** | Missed nuance; poor decisions | Expected loss (Bayesian) |

---

### Project Portfolio Connection

```
Project 1: Healthcare Readmissions ──────→ Domain expertise; clinical ML
Project 2: Marketing A/B Test ───────────→ Frequentist experimentation at scale
Project 3: Customer Segmentation ────────→ Unsupervised learning; RFM; cohort analysis
Project 4: Customer Churn Prediction ────→ Supervised ML; XGBoost; SHAP explainability
Project 5: Experimentation Framework ────→ Advanced testing; Bayesian; reusable toolkit
                                           (completes the analytical story)
```

All projects: [github.com/jordanshamu](https://github.com/jordanshamu)  
Portfolio: [datascienceportfol.io/jordanshamu](https://datascienceportfol.io/jordanshamu)


In [ ]:
print("✅ Experimentation Framework — Complete")
print()
print("Visualisations saved:")
for f in sorted(os.listdir(VIZ_DIR)):
    print(f"  {f}")
